In [ ]:
#| default_exp _utils_cam

In [ ]:
#| hide
from nbdev.showdoc import *

# Camera Utilities

> Utility functions and a basic ROS2 camera node for capturing and publishing video frames.

## Imports

The basic packages for camera work:

- `cv2` — OpenCV for camera capture and image processing
- `numpy` — numerical operations (images are numpy arrays)
- `matplotlib.pyplot` — plotting images and results

ROS2 packages:

- `rclpy` — ROS2 Python client library
- `Node` from `rclpy.node` — base class for ROS2 nodes
- `ExternalShutdownException` from `rclpy.executors` — clean shutdown handling
- `CvBridge` from `cv_bridge` — converts between ROS2 Image messages and OpenCV images
- `Image` from `sensor_msgs.msg` — ROS2 Image message type

In [ ]:
#| export
#| hide
import sys
import rclpy
from rclpy.node import Node
from rclpy.executors import ExternalShutdownException
import cv2
from cv_bridge import CvBridge
from sensor_msgs.msg import Image

## Camera Format Utilities

> **Note:** In OpenCV code you will often see the capture device stored in a variable named `cam` or `cap` — both are common conventions for `cv2.VideoCapture`. Throughout this notebook our utility functions use `cam` as parameter name, while the usage examples use `cap`. They refer to the same thing.

When connecting a camera through WSL or over limited bandwidth, sending raw uncompressed frames can cause timeouts and lag. These utility functions let you control the pixel/compression format (FOURCC) and resolution of a `cv2.VideoCapture` device.

In [ ]:
#| export
def set_cam_format_fourcc(cam, format='MJPG'):
    "Set the pixel/compression format (FOURCC) of a `cv2.VideoCapture` device. e.g. `'MJPG'`, `'YUYV'`, `'NV12'`."
    valid_formats = ['MJPG', 'YUYV', 'RGB3', 'GRAY', 'YUV420P', 'NV12', 'NV21']
    if format not in valid_formats:
        raise ValueError(f"Invalid format '{format}'. Valid formats are: {valid_formats}")
    cam.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*format))

def get_cam_format_fourcc(cam):
    "Get the current pixel/compression format (FOURCC) of a `cv2.VideoCapture` device."
    codec_int = int(cam.get(cv2.CAP_PROP_FOURCC))
    codec_str = "".join([chr((codec_int >> 8 * i) & 0xFF) for i in range(4)])
    return codec_str

def set_cam_resolution(cam, width=640, height=480):
    "Set the capture resolution of a `cv2.VideoCapture` device."
    cam.set(cv2.CAP_PROP_FRAME_WIDTH, width)
    cam.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

def get_cam_resolution(cam):
    "Get the current capture resolution of a `cv2.VideoCapture` device."
    width = int(cam.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cam.get(cv2.CAP_PROP_FRAME_HEIGHT))
    return width, height

## Connecting to a Camera

To connect to a camera, use `cv2.VideoCapture`. The source can be:

- An integer (e.g. `0`) for a USB camera
- A URL string for a network stream (e.g. `"http://192.168.1.100:8080/video"`)

```python
source = "0"
cap = cv2.VideoCapture(int(source) if source.isdigit() else source)
```

Check and set the compression format — MJPG is recommended for bandwidth-limited scenarios like WSL:

```python
print("Current format:", get_cam_format_fourcc(cap))
set_cam_format_fourcc(cap, 'MJPG')
print("New format:", get_cam_format_fourcc(cap))
```

Capture and display a frame:

```python
ret, frame = cap.read()
fig, ax = plt.subplots()
ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
ax.set_title(f"Captured Image. Resolution: {frame.shape[1]}x{frame.shape[0]}")
ax.axis('off')
plt.show()
```

![Single frame captured from USB camera](images/01_simple_example/single_capture.png)

## Resolution and Resizing

You can change the capture resolution with `set_cam_resolution`, or resize frames after capture with `cv2.resize`. When bandwidth is limited, aim to send the smallest frame possible.

```python
HD, LD = (1280, 720), (640, 480)

set_cam_resolution(cap, *HD)
ret, hd_frame = cap.read()

set_cam_resolution(cap, *LD)
ret, ld_frame = cap.read()

# Post-processing: resize between resolutions
hd_to_ld = cv2.resize(hd_frame, LD)
ld_to_hd = cv2.resize(ld_frame, HD)
```

The comparison below shows the difference between native resolution and resized frames:

![HD vs LD resolution comparison — native capture vs resized frames](images/01_simple_example/hd_ld_comparison.png)

> Always release the camera when done: `cap.release()`

## CameraNode

`CameraNode` is a ROS2 node that captures frames from a camera and publishes them as `sensor_msgs/Image` messages. It supports the following parameters:

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `video_source` | string | `'0'` | Camera index or stream URL |
| `frame_rate` | float | `30.0` | Publishing frequency in FPS |
| `topic_name` | string | `'/camera/image_raw'` | Topic to publish images on |
| `video_fourcc` | string | `'MJPG'` | Compression format |
| `resolution_width` | int | `640` | Output frame width |
| `resolution_height` | int | `480` | Output frame height |

If the camera's native resolution differs from the requested resolution, frames are automatically resized before publishing.

In [ ]:
#| export
class CameraNode(Node):
    "ROS2 node that captures camera frames and publishes them as `sensor_msgs/Image`."
    def __init__(self):
        super().__init__('camera_node')

        # Declare parameters with defaults
        self.declare_parameter('video_source', '0')
        self.declare_parameter('frame_rate', 30.0)
        self.declare_parameter('topic_name', '/camera/image_raw')
        self.declare_parameter('video_fourcc', 'MJPG')
        self.declare_parameter('resolution_width', 640)
        self.declare_parameter('resolution_height', 480)

        # Read parameter values
        self.source = self.get_parameter('video_source').value
        self.frame_rate = self.get_parameter('frame_rate').value
        self.topic_name = self.get_parameter('topic_name').value
        self.fourcc = self.get_parameter('video_fourcc').value
        self.resolution = (
            self.get_parameter('resolution_width').value,
            self.get_parameter('resolution_height').value
        )

        assert isinstance(self.source, str), "video_source must be a string"
        assert isinstance(self.frame_rate, (int, float)), 'frame_rate must be a number'
        assert isinstance(self.topic_name, str), 'topic_name must be a string'
        assert isinstance(self.fourcc, str), 'video_fourcc must be a string'
        assert isinstance(self.resolution, tuple) and len(self.resolution) == 2, \
            'resolution must be a tuple of two elements'

        # Open camera
        if self.source.isdigit():
            self.cap = cv2.VideoCapture(int(self.source))
        else:
            self.cap = cv2.VideoCapture(self.source)

        if not self.cap.isOpened():
            self.get_logger().error(f"Failed to open source: {self.source}")
            raise RuntimeError('Could not open video source')

        # Set compression format (critical for WSL bandwidth)
        set_cam_format_fourcc(self.cap, format=self.fourcc)

        # Publisher
        queue_size = 20
        self.publisher_ = self.create_publisher(Image, self.topic_name, queue_size)

        # Timer for periodic frame capture
        timer_period = 1.0 / self.frame_rate
        self.timer = self.create_timer(timer_period, self.timer_callback)

        # CvBridge for OpenCV <-> ROS2 Image conversion
        self.bridge = CvBridge()
        self.frame_count = 0

        self.get_logger().info(
            f'Publishing at {self.frame_rate} FPS from source {self.source} '
            f'on "{self.topic_name}"'
        )

    def timer_callback(self):
        "Capture a frame, resize if needed, and publish as a ROS2 Image message."
        ret, frame = self.cap.read()
        if not ret:
            self.get_logger().warn('Failed to read frame from camera')
            return

        # Resize if camera resolution differs from target
        if (frame.shape[1], frame.shape[0]) != self.resolution:
            frame = cv2.resize(frame, self.resolution, interpolation=cv2.INTER_CUBIC)

        msg = self.bridge.cv2_to_imgmsg(frame, encoding='bgr8')
        self.publisher_.publish(msg)

        if self.frame_count % 10 == 0:
            self.get_logger().info(f'Published frame {self.frame_count}')
        self.frame_count += 1

    def destroy_node(self):
        "Release the camera before destroying the node."
        self.cap.release()
        super().destroy_node()

### Timer Callback

To stream a video feed, we need to capture images at a certain rate (FPS) and publish each one. ROS2 provides a **timer** object that calls a callback function at a fixed period — in our case, every `1 / frame_rate` seconds. Inside that callback we capture a frame, resize it if needed, and publish it.

In [ ]:
#| echo: false
import inspect

In [ ]:
#| echo: false
print(inspect.getsource(CameraNode.timer_callback))

### Destroyer

When shutting down the node, we need to release the camera resource. The `destroy_node` method is overridden to ensure `self.cap.release()` is called when the node is destroyed. This prevents resource leaks and ensures the camera can be accessed by other applications after shutdown.

In [ ]:
show_doc(CameraNode.destroy_node)

## Entry Point

Every ROS2 node file must define a `main()` function — this is how `ros2 run` discovers and launches the node.

In [ ]:
#| export
def main(args=None):
    "Initialize ROS2, spin the `CameraNode`, and handle shutdown."
    rclpy.init(args=args)
    node = CameraNode()

    try:
        rclpy.spin(node)
    except (KeyboardInterrupt, ExternalShutdownException):
        node.get_logger().info('Node stopped by user or external shutdown')
    finally:
        node.destroy_node()
        rclpy.try_shutdown()

In [ ]:
#| exporti
if not hasattr(sys, 'ps1'):
    if __name__ == "__main__":
        main()

> The `sys.ps1` check prevents the node from launching when the module is imported in an interactive session (notebook or REPL). This way, `python3 _utils_cam.py` starts the node, but `import _utils_cam` in a notebook does not.

## Usage

First, export the notebook to generate the Python module:

```bash
uv run nbdev-export
```

This creates `ros2_cam_nbdev/_utils_cam.py`.

### Run the node

**Terminal 1** — run the node:

```bash
uv run python -m ros2_cam_nbdev._utils_cam
```

**Terminal 2** — check the topic is publishing:

```bash
ros2 topic hz /camera/image_raw
```

See `08_visualization` for tools to view the images.

> **Note:** This module starts with `_` so it cannot be compiled as a ROS2 entry
> point. To run a node via `ros2 run`, its filename must not start with `_`.
> See `02_standard_publisher` for an example with both methods.

## Visualize

With the node running, open a second terminal and visualize the images:

```sh
ros2 run rqt_image_view rqt_image_view
```

Select `/camera/image_raw` from the dropdown.

For more visualization options (rviz2, Foxglove), see `08_visualization`.

## Next Steps

This notebook was an introduction to building a ROS2 node that uses OpenCV to connect to a camera, capture frames, and publish them on a ROS2 topic so other nodes can consume those images.

In upcoming notebooks we will incorporate optimizations and best practices:

- **Runtime configuration** — Once compiled, ROS2 nodes can be reconfigured (parameters, topic names, node name) without changing code, either via CLI arguments or through config files and launch files.
- **Automation scripts** — Workflows and scripts that handle workspace creation, package scaffolding with correct dependencies, and auto-generation of `setup.py` and `package.xml` — the files responsible for registering nodes, entry points, and non-code assets (robot models, maps, config files, etc.).

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()